# Hari 001 — Return harian & kumulatif

| Hari | Tanggal | Jenis | Topik |
|------|---------|-------|-------|
| 001 | 2026-11-19 | `quant` | Return harian & kumulatif |

> Bagian dari **100QB** — 100 Days of Quant + Kebun Simulasi.

## Konsep

**Simple return** (return sederhana) mengukur perubahan persentase harga dari satu periode ke periode berikutnya:

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

**Return kumulatif** mengukur total pertumbuhan nilai investasi dari awal periode hingga waktu $t$ dengan memperhitungkan efek *compounding* (bunga majemuk):

$$R_{\text{cum}, t} = \prod_{i=1}^{t} (1 + R_i) - 1 = \frac{P_t}{P_0} - 1$$

**Kenapa dipakai:**
- Harga saham mentah sulit dibandingkan antar aset (misalnya harga dalam ribuan rupiah vs ratusan dolar); return menormalkan skala sehingga performa aset dapat dibandingkan secara adil.
- Return harian bersifat *scale-free* dan menjadi dasar perhitungan risiko (volatilitas), *Sharpe ratio*, serta evaluasi portofolio.

## Setup

In [1]:
import sys
from pathlib import Path

# Cari root repo supaya package `qb100` di src/ bisa di-import dari mana pun
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "qb100").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qb100 import data, finance as fin, plotting, simulation as sim

plotting.set_style()
rng = sim.get_rng(42)

TICKERS = ["BBCA.JK", "^JKSE", "AAPL"]
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"


## Data

Kita mengambil harga penutupan (*Close*) untuk tiga aset: `BBCA.JK` (Bank Central Asia), `^JKSE` (IHSG), dan `AAPL` (Apple) menggunakan `qb100.data.get_close`. Bila koneksi internet/API `yfinance` gagal atau *offline*, kode otomatis jatuh balik ke data harga simulasi menggunakan `qb100.simulation.random_walk`.

In [2]:
try:
    prices = data.get_close(TICKERS, start=START_DATE, end=END_DATE)
    prices = prices.ffill().dropna()
    data_source = "yfinance (cache/online)"
except Exception as exc:
    print(f"Gagal mengunduh dari yfinance ({exc}); memakai data simulasi.")
    dates = pd.bdate_range(START_DATE, END_DATE)
    n_steps = len(dates) - 1
    log_paths = sim.random_walk(
        n_steps=n_steps,
        n_paths=len(TICKERS),
        step_std=0.014,
        drift=0.0004,
        start=0.0,
        seed=rng,
    )
    base_prices = np.array([9500.0, 7200.0, 185.0])
    prices = pd.DataFrame(base_prices * np.exp(log_paths), index=dates, columns=TICKERS)
    data_source = "simulasi (random_walk)"

print(f"Sumber data: {data_source} | Baris: {len(prices)}")
prices.head()

Sumber data: yfinance (cache/online) | Baris: 258


,BBCA.JK,^JKSE,AAPL
Date,,,
2024-01-02,8404.843750,7323.587891,183.264771
2024-01-03,8337.960938,7279.090820,181.892548
2024-01-04,8449.430664,7359.763184,179.582474
2024-01-05,8538.606445,7350.619141,178.861801
2024-01-08,8538.606445,7283.575195,183.185791


## Implementasi

In [3]:
# 1. Hitung simple return harian menggunakan qb100.finance.simple_returns
daily_ret = fin.simple_returns(prices)
print("5 hari pertama return harian (%):")
(daily_ret * 100).round(3).head()

5 hari pertama return harian (%):


,BBCA.JK,^JKSE,AAPL
Date,,,
2024-01-03,-0.796,-0.608,-0.749
2024-01-04,1.337,1.108,-1.270
2024-01-05,1.055,-0.124,-0.401
2024-01-08,0.000,-0.912,2.418
2024-01-09,0.522,-1.145,-0.226


In [4]:
# 2. Hitung return kumulatif menggunakan qb100.finance.cumulative_returns
cum_ret = fin.cumulative_returns(daily_ret)
print("5 hari terakhir return kumulatif (%):")
(cum_ret * 100).round(2).tail()

5 hari terakhir return kumulatif (%):


,BBCA.JK,^JKSE,AAPL
Date,,,
2024-12-23,6.63,-3.10,38.18
2024-12-24,6.36,-3.52,39.77
2024-12-26,6.36,-3.52,40.21
2024-12-27,6.91,-3.92,38.35
2024-12-30,5.54,-3.33,36.52


In [5]:
# 3. Ringkasan performa tiap aset
summary = pd.DataFrame({
    "Mean Harian (%)": daily_ret.mean() * 100,
    "Volatilitas Tahunan (%)": fin.volatility(daily_ret) * 100,
    "Sharpe Ratio": fin.sharpe_ratio(daily_ret),
    "Total Return Kumulatif (%)": cum_ret.iloc[-1] * 100,
    "Max Drawdown (%)": fin.max_drawdown(prices) * 100,
}).round(2)
summary

,Mean Harian (%),Volatilitas Tahunan (%),Sharpe Ratio,Total Return Kumulatif (%),Max Drawdown (%)
BBCA.JK,0.03,21.34,0.35,5.54,-11.43
^JKSE,-0.01,12.54,-0.20,-3.33,-11.74
AAPL,0.13,22.14,1.49,36.52,-15.35


## Visualisasi

In [6]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Panel 1: Harga ternormalisasi (basis 100 di hari pertama)
norm_prices = prices / prices.iloc[0] * 100
norm_prices.plot(ax=ax1, linewidth=1.8)
ax1.axhline(100, color="gray", linestyle=":", linewidth=1)
ax1.set_title("Perbandingan Harga Ternormalisasi (Basis = 100)")
ax1.set_ylabel("Indeks (Basis 100)")
ax1.legend(loc="upper left")

# Panel 2: Return kumulatif (%)
(cum_ret * 100).plot(ax=ax2, linewidth=1.8)
ax2.axhline(0, color="gray", linestyle=":", linewidth=1)
ax2.set_title("Return Kumulatif Harian (%)")
ax2.set_ylabel("Return Kumulatif (%)")
ax2.set_xlabel("Tanggal")
ax2.legend(loc="upper left")

fig.tight_layout()
plotting.save_figure(fig, "day001_return-harian-kumulatif")
plt.show()

/var/folders/ms/5n6sfq8j2z52l34_szzbz02w0000gn/T/ipykernel_46180/4198750806.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Kesimpulan (3 poin)

1. **Simple return** menormalkan perubahan harga antar periode sehingga aset dengan skala harga berbeda (seperti `BBCA.JK` dalam IDR dan `AAPL` dalam USD) bisa dibandingkan langsung.
2. **Return kumulatif** menggabungkan return harian secara multiplikatif ($\prod (1 + R_t) - 1$), bukan dijumlahkan begitu saja, karena ada efek *compounding*.
3. Nilai return kumulatif pada waktu $t$ selalu identik dengan perubahan harga ternormalisasi dari basis awal ($P_t / P_0 - 1$), yang terlihat jelas pada kesamaan bentuk kedua kurva grafik.

## Referensi

- Tsay, R. S. (2010). *Analysis of Financial Time Series* (Bab 1: Financial Data and Their Properties).
- Dokumentasi `pandas.DataFrame.pct_change` dan `cumprod` — https://pandas.pydata.org/docs/
- Hilpisch, Y. (2018). *Python for Finance* — O'Reilly Media.